<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/00_quickstart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

	# 배터리 열폭주 이미지 분류 · Colab 퀵스타트

	## 0. GPU 확인

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

	## 1. 코드 내려받기

In [ ]:
!git clone https://github.com/Hwk040319/MJY-ML.git
%cd MJY-ML
!pip install -q -r requirements.txt

	## 2. 데이터 내려받기



In [ ]:
!pip install -q gdown

# 1회차 강의 실습용 (약 700장, 22MB)
FILE_ID = '13DWY5tg_L4SYxujkdVQ89qEQZC08lr7L'

# 회차 사이 팀 실험용 (전체, 11GB) — 위 줄을 주석 처리하고 아래를 사용
# FILE_ID = '1PJNyDDdYd47wXD83DiW9PqFzbe7TLl0n'

!gdown "https://drive.google.com/uc?id=$FILE_ID" -O data.tar
!mkdir -p data && tar -xf data.tar -C data
!ls data

## 3. 데이터 검사




In [ ]:
!python check_data.py --data-root data

## 4. Baseline 학습

In [ ]:
!python train_baseline.py \
  --data-root data \
  --output-dir outputs/baseline \
  --epochs 3 \
  --batch-size 32 \
  --lr 1e-3

	## 5. 결과 확인

In [ ]:
import json
with open('outputs/baseline/validation_report.json', encoding='utf-8') as f:
    report = json.load(f)
print('Macro F1:', round(report['macro_f1'], 4))
print('Accuracy:', round(report['accuracy'], 4))
print('Class F1 (초기/중기/후기):', [round(v, 4) for v in report['class_f1']])

## 6. 개선 실험 (회차 사이)

In [ ]:
# A. 데이터 증강
!python train_baseline.py --data-root data --augment --output-dir outputs/exp_aug

# B. 클래스 가중치
!python train_baseline.py --data-root data --use-class-weights --output-dir outputs/exp_weight

# C. 전체 미세조정 (작은 learning rate 필수)
!python train_baseline.py --data-root data --unfreeze --lr 1e-4 --output-dir outputs/exp_ft

## 7. 최종 제출 · 2회차 전날 14:00 마감

In [ ]:
import torch
FINAL = 'outputs/exp_aug/best_model.pt'   # 최종 선택한 경로로 변경
ckpt = torch.load(FINAL, map_location='cpu', weights_only=False)
print('Public Val Macro F1:', round(ckpt['macro_f1'], 4), '| epoch:', ckpt['epoch'])

from google.colab import files
files.download(FINAL)
# [팀명]_best_model.pt 로 이름 변경 후 제출 폼에 업로드